In [1]:
# read final_df from csv file
import pandas as pd
df = pd.read_csv(r'd:/Github/Data/data/material_weight_data_20250114.txt')
# exclude historical buildings before 1930
df = df[df['year of construction of the building yyyy'] >= 1930]


C:\Users\xiong\AppData\Local\Temp\ipykernel_3260\3526432643.py:3: DtypeWarning: Columns (18,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'd:/Github/Data/data/material_weight_data_20250114.txt')


In [2]:
# --- 1. Merge stainless steel into steel ---
# Example: if you have both "stainless steel_kitchen sink material weights"
# and "steel_air duct material weights", add them into a single "steel_*" column

# Let's assume you want to merge all stainless steel into steel (column names contain "stainless steel_")
steel_cols = [col for col in df.columns if col.startswith("steel_")]
stainless_cols = [col for col in df.columns if col.startswith("stainless steel_")]

for s_col in stainless_cols:
    # map "stainless steel_xxx" -> "steel_xxx"
    steel_col = "steel_" + s_col.replace("stainless steel_", "")
    if steel_col in df.columns:
        df[steel_col] = df[steel_col].fillna(0) + df[s_col].fillna(0)
    else:
        df[steel_col] = df[s_col]
    df.drop(columns=s_col, inplace=True)  # drop stainless column

# --- 2. Remove bathtub components for non-residential buildings ---
bathtub_cols = [col for col in df.columns if "bathtub" in col]

# condition: only keep bathtubs if building_class == "residential"
non_res_mask = df["building class"] != "residential"
df.loc[non_res_mask, bathtub_cols] = 0

In [3]:

# --- Step 1: Normalize stainless steel into steel ---
df = df.rename(columns=lambda x: x.replace("stainless steel", "steel"))

# --- Step 2: Build a mapping from column -> material ---
material_map = {}
for col in df.columns:
    if "material weights" in col:
        mat = col.split("_")[0]  # take first part before "_"
        material_map[col] = mat

# --- Step 3: Aggregate by material ---
material_sums = df[list(material_map.keys())].groupby(material_map, axis=1).sum()

# --- Step 4: Merge with building info ---
df_agg = pd.concat([df.drop(columns=material_map.keys()), material_sums], axis=1)

C:\Users\xiong\AppData\Local\Temp\ipykernel_3260\374061674.py:12: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  material_sums = df[list(material_map.keys())].groupby(material_map, axis=1).sum()


In [4]:
import pandas as pd
import numpy as np

# ---- config ----
area_col = "building area"          # change if yours differs
id_col = "federal building identifier"  # optional

# factors = {
#     "steel": 2.0,
#     "aluminum": 9.0,
#     "copper": 4.0,
#     "brass": 3.0,
#     "cast iron": 1.9,
#     "pvc": 2.5,
#     "plastic": 2.5,
#     "pex": 2.0,
#     "polyurethane foam": 5.0,
#     "fiberglass": 1.4,
#     "mineral wool": 1.2,
#     "porcelain": 1.0,
#     "zinc coating": 3.5,
#     "acrylic": 3.0,
#     "refrigerant": 1430.0,
#     "other electronics": 15.0,
# }
# Per-unit GHG emission factors (kgCO2-eq per kg) from Table S9
factors = {
    "steel": 2.4,
    "aluminum": 10.0,
    "copper": 6.9,
    "brass": 5.7,
    "cast iron": 1.9,
    "pvc": 1.7,
    "other plastics": 0.6,
    "pex": 0.8,
    "polyurethane foam": 4.6,
    "fiberglass": 2.5,
    "mineral wool": 1.6,
    "porcelain": 2.0,
    "zinc coating": 0.6,
    "acrylic": 1.1,
    "polyester resin": 2.5,
    "refrigerant": 17.1,
    "other electronics": 55.0,
}


# ensure numeric weights
for mat in factors.keys():
    if mat in df_agg.columns:
        df_agg[mat] = pd.to_numeric(df_agg[mat], errors="coerce")

# compute per-material GHG columns
ghg_cols = []
for mat, ef in factors.items():
    if mat in df_agg.columns:
        col = f"{mat}_GHG"
        df_agg[col] = df_agg[mat].fillna(0) * ef
        ghg_cols.append(col)

# sum to total and normalize by area
df_agg["Total_GHG_kgCO2"] = df_agg[ghg_cols].sum(axis=1)
df_agg["kgCO2_per_m2"] = np.where(
    pd.to_numeric(df_agg[area_col], errors="coerce") > 0,
    df_agg["Total_GHG_kgCO2"] / pd.to_numeric(df_agg[area_col], errors="coerce"),
    np.nan
)

# (optional) keep a tidy table of the essentials
cols_to_show = [c for c in [id_col, area_col, "Total_GHG_kgCO2", "kgCO2_per_m2"] if c in df_agg.columns]
result = df_agg[cols_to_show].copy()


In [9]:
lower = df_agg["kgCO2_per_m2"].quantile(0.10)   # 1st percentile
upper = df_agg["kgCO2_per_m2"].quantile(0.90)   # 99th percentile

df_filtered = df_agg[
    (df_agg["kgCO2_per_m2"] >= lower) &
    (df_agg["kgCO2_per_m2"] <= upper)
]

# Group stats
group_stats = df_filtered.groupby("building class")["kgCO2_per_m2"].describe()
print(group_stats)



                    count        mean         std       min        25%  \
building class                                                           
commercial        71014.0  111.164633  105.511654  7.390868  26.567913   
institutional     18341.0   57.600835   66.220637  7.396119  17.483948   
other            707883.0  144.067385  107.294326  7.390823  47.845476   
residential     1353724.0   21.395165   17.382057  7.390548  11.521116   

                       50%         75%         max  
building class                                      
commercial       67.141257  176.733803  386.086465  
institutional    32.478335   67.714496  385.712177  
other           123.143777  221.752272  386.121187  
residential      15.840187   25.577570  385.607826  


## *Test GHG calculation*

In [ ]:
from brightway2 import *
from bw2io.importers.ecospold2 import SingleOutputEcospold2Importer

# Set up Brightway2 project
projects.set_current("GHG Material Analysis")
bw2setup()

In [ ]:
ei = SingleOutputEcospold2Importer(
    dirpath="d:/Github/Data/ecoinvent 3.11_cutoff_ecoSpold02/datasets",  # Replace with your path to ecoSpold02 files
    db_name="ecoinvent_3_11_cutoff",  # Name the database
)
ei.apply_strategies()
ei.write_database()

Extracting XML data from 25412 datasets
16:14:01 [info     ] Extracted 25412 datasets in 41.01 seconds
Applying strategy: normalize_units
Applying strategy: update_ecoinvent_locations
Applying strategy: remove_zero_amount_coproducts
Applying strategy: remove_zero_amount_inputs_with_no_activity
Applying strategy: remove_unnamed_parameters
Applying strategy: es2_assign_only_product_with_amount_as_reference_product
Applying strategy: assign_single_product_as_activity
Applying strategy: create_composite_code
Applying strategy: drop_unspecified_subcategories
Applying strategy: fix_ecoinvent_flows_pre35
Applying strategy: drop_temporary_outdated_biosphere_flows
Applying strategy: link_biosphere_by_flow_uuid
Applying strategy: link_internal_technosphere_by_composite_code
Applying strategy: delete_exchanges_missing_activity
Applying strategy: delete_ghost_exchanges
Applying strategy: remove_uncertainty_from_negative_loss_exchanges
Applying strategy: fix_unreasonably_high_lognormal_uncertaintie

100%|██████████| 25412/25412 [01:37<00:00, 260.31it/s]


16:15:50 [info     ] Vacuuming database            
Created database: ecoinvent_3_11_cutoff


Brightway2 SQLiteBackend: ecoinvent_3_11_cutoff

In [ ]:
# materials_mapping = {
#     "acrylic": "market for acrylic resin",
#     "aluminum": "market for aluminium, primary",
#     "brass": "market for brass",
#     "cast iron": "market for cast iron",
#     "copper": "market for copper",
#     "fiberglass": "market for glass fibre",
#     "mineral wool": "market for mineral wool",
#     "other electronics": "market for electronic component",
#     "pex": "market for polyethylene, high density, granulate",
#     "other plastics": "market for plastic, unspecified",
#     "polyurethane foam": "market for polyurethane, rigid foam",
#     "porcelain": "market for ceramics",
#     "pvc": "market for polyvinylchloride, bulk polymerized",
#     "refrigerant": "market for refrigerant R134a",
#     "steel": "market for steel, low-alloyed",
#     "zinc coating": "market for zinc coating",
}

In [ ]:
# import pandas as pd

# # Load your material data as a DataFrame
# data = material_weights

# # Function to calculate GHG emissions
# def calculate_ghg(material, weight_kilotons):
#     db = Database("ecoinvent_3_11_cutoff")
#     if material not in materials_mapping:
#         return None  # Return None if material is not mapped
    
#     try:
#         # Get the corresponding Ecoinvent process
#         process = db.get(materials_mapping[material])
#         # Define the functional unit (convert kilotons to kg)
#         functional_unit = {process: weight_kilotons * 1e6}
#         # Perform LCA
#         lca = LCA(functional_unit)
#         lca.lci()
#         lca.lcia()
#         return lca.score  # Return GHG emissions
#     except Exception as e:
#         print(f"Error calculating GHG for {material}: {e}")
#         return None

# # Apply the calculation to each row
# data["calculated_GHG"] = data.apply(
#     lambda row: calculate_ghg(row["material"], row["material_weight_kilotons"]),
#     axis=1
# )

# # Save the results
# data.to_csv("material_ghg_results.csv", index=False)

In [ ]:
# def calculate_ghg(material, weight_kilotons):
#     process = db.get(materials_mapping[material])
#     functional_unit = {process: weight_kilotons * 1e6}  # Convert kilotons to kg
#     lca = LCA(functional_unit)
#     lca.lci()
#     lca.lcia()
#     return lca.score

In [ ]:
# import pandas as pd

# data = pd.read_csv("material_weights_20250106.csv")
# data["calculated_GHG"] = data.apply(
#     lambda row: calculate_ghg(row["material"], row["material_v_kilotons"]),
#     axis=1,
# )
